# Install Instructions

I will assume that you have Julia installed on your local machine. (If not go to the Julia website: https://julialang.org/ and use the "Download" button.)

### 1. Install IJulia

I could put instructions here but it is probably better to just look at the github readme here: https://github.com/JuliaLang/IJulia.jl or the installation instructions in the documentation here: https://julialang.github.io/IJulia.jl/stable/manual/installation/

### 2. Launch the Notebook and Run

Launch the notebook and run cells as you would any other jupyter notebook.

NOTE: The first cell below will create a Julia environment in the same directory as this notebook resides in. The second cell will install the needed packages.

Progressive Hedging Repo: https://github.com/NREL/ProgressiveHedging.jl
Other JuMP Compatible Optimizers: https://jump.dev/JuMP.jl/stable/installation/#Install-a-solver

In [1]:
import Pkg
Pkg.activate(".")
Pkg.status()

  Activating project at `~/Desktop/qccbo_proj/qsaa`


Status `~/Desktop/qccbo_proj/qsaa/Project.toml`
  [87dc4568] HiGHS v1.5.2
  [b6b21f68] Ipopt v1.4.1
⌃ [4076af6c] JuMP v1.12.0
Info Packages marked with ⌃ have new versions available and may be upgradable.


In [2]:
Pkg.add("JuMP")
Pkg.add("HiGHS")
Pkg.add("Ipopt")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/Desktop/qccbo_proj/qsaa/Project.toml`
  No Changes to `~/Desktop/qccbo_proj/qsaa/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/qccbo_proj/qsaa/Project.toml`
  No Changes to `~/Desktop/qccbo_proj/qsaa/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/qccbo_proj/qsaa/Project.toml`
  No Changes to `~/Desktop/qccbo_proj/qsaa/Manifest.toml`


In [3]:
import HiGHS
import Ipopt
import JuMP

# Problem as Described

## Parameters

In [17]:
c_x = 2.0
c_w = 0.0
c_y = 10.0
thermal_min = 0
thermal_max = 3
wind_min = 0
wind_max = 3
# wind_set = collect(0:3)
d = 2
ξ = collect(wind_min:1:wind_max)
NSCEN = length(ξ)
prob = [0.1, 0.2, 0.3, 0.4] # prob[s] gives probability of ξ taking value ξ[s]
# prob = ones(NSCEN) ./ NSCEN # Uniform (discrete) probabilities

4-element Vector{Float64}:
 0.1
 0.2
 0.3
 0.4

In [5]:
# MY VERSION
c_x = 3.0
c_w = 1.0
c_y = 10.0
thermal_min = 0
thermal_max = 3
wind_min = 0
wind_max = 3
# wind_set = collect(0:3)
d = 3
ξ = collect(wind_min:1:wind_max)
NSCEN = length(ξ)
prob = [0.1,0.2,0.3,0.4]
#prob = [0.25, 0.25, 0.0, 0.5] # prob[s] gives probability of ξ taking value ξ[s]
# prob = ones(NSCEN) ./ NSCEN # Uniform (discrete) probabilities

4-element Vector{Float64}:
 0.1
 0.2
 0.3
 0.4

## Optimization Problem

In [6]:
# Create model
m = JuMP.Model(HiGHS.Optimizer)

A JuMP Model
Feasibility problem with:
Variables: 0
Model mode: AUTOMATIC
CachingOptimizer state: EMPTY_OPTIMIZER
Solver name: HiGHS

In [7]:
# Add Variables
JuMP.@variable(m, thermal_min <= x <= thermal_max, Int)
JuMP.@variable(m, wind_min <= w[1:NSCEN] <= wind_max, Int)
JuMP.@variable(m, 0 <= y_minus[1:NSCEN] <= d, Int)
;

In [8]:
# Add Constraints
JuMP.@constraint(m, x .+ w .+ y_minus .== d)
JuMP.@constraint(m, w .<= ξ)
;

In [9]:
# Add Objective
obj_expr = zero(JuMP.AffExpr)
JuMP.add_to_expression!(obj_expr, c_x, x) # Thermal cost
for s in 1:NSCEN
    JuMP.add_to_expression!(obj_expr, prob[s]*c_w, w[s]) # Wind cost -- c_w == 0
    JuMP.add_to_expression!(obj_expr, prob[s]*c_y, y_minus[s]) # Lost load
end
JuMP.@objective(m, Min, obj_expr)
;

In [10]:
# For ProgressiveHedging.jl -- non-extensive form
# Also requires a scenario tree
# m = JuMP.Model()
# m = JuMP.@variable(m, x_s) # <---- first stage variable
# PH enforces x_s == x_t for all s,t ∈ S
# m = JuMP.@variable(m, w_s)
# JuMP.@constraint(m, w <= ξ_s)
# 0----0
# |____0
# |____0

In [11]:
# function create_single_scenario_model(scen_number::Int)

In [12]:
println(m)
;

Min 3 x + 0.1 w[1] + y_minus[1] + 0.2 w[2] + 2 y_minus[2] + 0.3 w[3] + 3 y_minus[3] + 0.4 w[4] + 4 y_minus[4]
Subject to
 x + w[1] + y_minus[1] = 3
 x + w[2] + y_minus[2] = 3
 x + w[3] + y_minus[3] = 3
 x + w[4] + y_minus[4] = 3
 w[1] ≤ 0
 w[2] ≤ 1
 w[3] ≤ 2
 w[4] ≤ 3
 x ≥ 0
 w[1] ≥ 0
 w[2] ≥ 0
 w[3] ≥ 0
 w[4] ≥ 0
 y_minus[1] ≥ 0
 y_minus[2] ≥ 0
 y_minus[3] ≥ 0
 y_minus[4] ≥ 0
 x ≤ 3
 w[1] ≤ 3
 w[2] ≤ 3
 w[3] ≤ 3
 w[4] ≤ 3
 y_minus[1] ≤ 3
 y_minus[2] ≤ 3
 y_minus[3] ≤ 3
 y_minus[4] ≤ 3
 x integer
 w[1] integer
 w[2] integer
 w[3] integer
 w[4] integer
 y_minus[1] integer
 y_minus[2] integer
 y_minus[3] integer
 y_minus[4] integer



## Solution

In [13]:
JuMP.optimize!(m)
;

Running HiGHS 1.5.3 [date: 1970-01-01, git hash: 45a127b78]
Copyright (c) 2023 HiGHS under MIT licence terms
Presolving model
3 rows, 7 cols, 9 nonzeros
2 rows, 5 cols, 6 nonzeros
2 rows, 5 cols, 6 nonzeros
Objective function is integral with scale 10

Solving MIP model with:
   2 rows
   5 cols (1 binary, 4 integer, 0 implied int., 0 continuous)
   6 nonzeros

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
     Proc. InQueue |  Leaves   Expl. | BestBound       BestSol              Gap |   Cuts   InLp Confl. | LpIters     Time

         0       0         0   0.00%   4.2             inf                  inf        0      0      0         0     0.0s

Solving report
  Status            Optimal
  Primal bound      7.9
  Dual bound        7.9
  Gap               0% (tolerance: 0.01%)
  Solution status   feasible
                    7.9 (objective)
                    0 (bound viol.)
                    0 (int. viol.)

In [14]:
# @show JuMP.termination_status(m)
# @assert JuMP.termination_status(m) == JuMP.OPTIMAL # Throw an error if solve failed
# @show JuMP.objective_value(m)
# ;
JuMP.solution_summary(m)

* Solver : HiGHS

* Status
  Result count       : 1
  Termination status : OPTIMAL
  Message from the solver:
  "kHighsModelStatusOptimal"

* Candidate solution (result #1)
  Primal status      : FEASIBLE_POINT
  Dual status        : NO_SOLUTION
  Objective value    : 7.90000e+00
  Objective bound    : 7.90000e+00
  Relative gap       : 0.00000e+00

* Work counters
  Solve time (sec)   : 3.34515e-02
  Simplex iterations : 2
  Barrier iterations : -1
  Node count         : 1


In [15]:
for var in JuMP.all_variables(m)
    println(var, " = ", JuMP.value(var))
end

x = 2.0
w[1] = 0.0
w[2] = 1.0
w[3] = 1.0
w[4] = 1.0
y_minus[1] = 1.0
y_minus[2] = 0.0
y_minus[3] = 0.0
y_minus[4] = 0.0


# Continuous Problem

In [ ]:
# # Uncomment below for a continuous version of the above 
# # problem (i.e. decision variables are continuous)
# NSCEN_C = NSCEN
# η = ξ
# prob_c = prob
# prob_c ./= sum(prob_c)

# Uncomment the below for using a continuous uniform random variable
NSCEN_C = 100
η = (wind_max - wind_min) * rand(NSCEN_C) .+ wind_min
# η = generate_normal(NSCEN_C, mu, sigma)
prob_c = ones(NSCEN_C) ./ NSCEN_C
;

In [ ]:
η

In [ ]:
# Create model
mc = JuMP.Model(Ipopt.Optimizer)
# JuMP.set_attribute(mc, "tol", 1e-12)
# JuMP.set_attribute(mc, "linear_solver", "ma57")

In [ ]:
# Add Variables
JuMP.@variable(mc, 0 <= x <= 3)
JuMP.@variable(mc, 0 <= w[1:NSCEN_C] <= 3)
JuMP.@variable(mc, 0 <= y_minus[1:NSCEN_C] <= d)
;

In [ ]:
# Add Constraints
JuMP.@constraint(mc, power_balance_c, x .+ w .+ y_minus .== d)
JuMP.@constraint(mc, wind_c, w .<= η)
;

In [ ]:
# Add Objective
obj_expr = zero(JuMP.AffExpr)
JuMP.add_to_expression!(obj_expr, c_x, x)
for s in 1:NSCEN_C
    JuMP.add_to_expression!(obj_expr, prob_c[s]*c_w, w[s])
    JuMP.add_to_expression!(obj_expr, prob_c[s]*c_y, y_minus[s])
end
JuMP.@objective(mc, Min, obj_expr)
;

In [ ]:
println(mc)
;

In [ ]:
JuMP.optimize!(mc)

In [ ]:
for var in JuMP.all_variables(mc)
    println(var, " = ", JuMP.value(var) )
end

In [ ]:
JuMP.dual.(power_balance_c)

In [ ]:
JuMP.dual.(wind_c)